# TOURGO Analytics Dashboard

Data source: TourGo real data from S3/Delta Lake.

This notebook contains the dashboard sections used for the final demo. Day 3 fills Sections 1-3 with Silver and Gold Delta tables.

In [0]:
# == SECTION 1: BUSINESS OVERVIEW ==
from pyspark.sql.functions import col, sum as _sum, avg, round as _round

spark.sql("USE tourgo")

df_summary_bookings = spark.read.table("silver_bookings")
df_summary_revenues = spark.read.table("silver_revenues")
df_summary_users = spark.read.table("silver_users")
df_summary_tours = spark.read.table("silver_tours")
df_summary_reviews = spark.read.table("silver_reviews")

total_bookings = df_summary_bookings.count()
confirmed_bookings = df_summary_bookings.filter(col("status") == "confirmed").count()
total_revenue = df_summary_revenues.agg(_sum("total_amount").alias("total_revenue")).collect()[0]["total_revenue"] or 0
total_customers = df_summary_users.filter(col("role") == "CUSTOMER").count()
active_tours = df_summary_tours.filter(col("status").isin("approved", "active", "published")).count()
avg_rating = df_summary_reviews.agg(_round(avg("rating"), 2).alias("avg_rating")).collect()[0]["avg_rating"] or 0

print("=" * 60)
print("SECTION 1: TOURGO BUSINESS OVERVIEW")
print("=" * 60)
print(f"Total Revenue      : {total_revenue:>15,.0f} VND")
print(f"Total Bookings     : {total_bookings:>15,}")
print(f"Confirmed Bookings : {confirmed_bookings:>15,}")
print(f"Customers          : {total_customers:>15,}")
print(f"Active Tours       : {active_tours:>15,}")
print(f"Average Rating     : {avg_rating:>15}")

df_booking_status = df_summary_bookings.groupBy("status").count().orderBy(col("count").desc())
display(df_booking_status)
# Databricks chart suggestion: bar chart, X = status, Y = count.

SECTION 1: TOURGO BUSINESS OVERVIEW
Total Revenue      :   6,387,390,000 VND
Total Bookings     :             103
Confirmed Bookings :              59
Customers          :               0
Active Tours       :             574
Average Rating     :            4.02


status,count
confirmed,59
cancelled,26
pending,18


In [0]:
# == SECTION 2: REVENUE ANALYSIS ==
df_rev_daily = spark.read.table("gold_revenue_daily")

print("=" * 60)
print("SECTION 2: REVENUE ANALYSIS")
print("=" * 60)
display(
    df_rev_daily
    .select("booking_date", "month", "total_revenue", "provider_revenue", "platform_fee", "num_transactions")
    .orderBy("booking_date")
)
# Databricks chart suggestion: line chart, X = booking_date, Y = total_revenue.

display(
    df_rev_daily
    .groupBy("month")
    .agg(
        _sum("total_revenue").alias("monthly_revenue"),
        _sum("provider_revenue").alias("monthly_provider_revenue"),
        _sum("platform_fee").alias("monthly_platform_fee"),
        _sum("num_transactions").alias("monthly_transactions")
    )
    .orderBy("month")
)
# Databricks chart suggestion: stacked/grouped bar chart by month.

SECTION 2: REVENUE ANALYSIS


booking_date,month,total_revenue,provider_revenue,platform_fee,num_transactions
null,null,5.69214E9,5.122926E9,5.69214E8,689
2026-06-05,2026-06,1.1E7,9900000.0,1100000.0,2
2026-06-07,2026-06,3000000.0,2700000.0,300000.0,1
2026-06-09,2026-06,1.0E7,9000000.0,1000000.0,1
2026-06-10,2026-06,1.965E7,1.7685E7,1965000.0,4
2026-06-11,2026-06,2.0E7,1.8E7,2000000.0,1
2026-06-12,2026-06,1500000.0,1350000.0,150000.0,1
2026-06-16,2026-06,5.05E7,4.545E7,5050000.0,2
2026-06-20,2026-06,350000.0,315000.0,35000.0,1
2026-06-21,2026-06,2000000.0,1800000.0,200000.0,1


Databricks visualization. Run in Databricks to view.

month,monthly_revenue,monthly_provider_revenue,monthly_platform_fee,monthly_transactions
null,5.69214E9,5.122926E9,5.69214E8,689
2026-06,1.18E8,1.062E8,1.18E7,14
2026-07,9.53E7,8.577E7,9530000.0,11
2026-08,7.41E7,6.669E7,7410000.0,18
2026-09,8.81E7,7.929E7,8810000.0,11
2026-10,7.35E7,6.615E7,7350000.0,11
2026-11,1.2585E8,1.13265E8,1.2585E7,11
2026-12,1.8E7,1.62E7,1800000.0,4
2027-01,1.5E7,1.35E7,1500000.0,6
2027-02,1400000.0,1260000.0,140000.0,3


In [0]:
# == SECTION 3: PROVIDER AND TOUR PERFORMANCE ==
df_providers = spark.read.table("gold_provider_performance")
df_tours = spark.read.table("gold_tour_performance")

print("=" * 60)
print("SECTION 3A: TOP PROVIDERS")
print("=" * 60)
display(
    df_providers
    .select(
        "creator_id",
        "total_provider_revenue",
        "total_gmv",
        "total_confirmed_bookings",
        "active_tours_count",
        "avg_rating"
    )
    .orderBy(col("total_provider_revenue").desc())
    .limit(10)
)
# Databricks chart suggestion: bar chart, X = creator_id, Y = total_provider_revenue.

print("=" * 60)
print("SECTION 3B: TOP TOURS")
print("=" * 60)
display(
    df_tours
    .select(
        "tour_id",
        "title",
        "category_names",
        "total_revenue",
        "total_bookings",
        "total_travelers",
        "avg_rating",
        "review_count"
    )
    .orderBy(col("total_revenue").desc())
    .limit(10)
)
# Databricks chart suggestion: bar chart, X = title, Y = total_revenue.

SECTION 3A: TOP PROVIDERS


creator_id,total_provider_revenue,total_gmv,total_confirmed_bookings,active_tours_count,avg_rating
139,6.444E7,7.16E7,4,3,null
150,3.33E7,3.7E7,4,3,4.5
181,2.565E7,2.85E7,2,2,4.0
156,2.529E7,2.81E7,4,2,3.0
196,2.169E7,2.41E7,3,2,4.0
211,2.16E7,2.4E7,2,1,3.0
192,1.89E7,2.1E7,1,1,null
164,1.8E7,2.0E7,1,1,3.0
160,1.6335E7,1.815E7,3,3,4.5
215,1.476E7,1.64E7,3,2,3.5


Databricks visualization. Run in Databricks to view.

SECTION 3B: TOP TOURS


tour_id,title,category_names,total_revenue,total_bookings,total_travelers,avg_rating,review_count
408,Phiêu lưu Hội An cuối tuần,Mạo hiểm,4.06E7,2,10,null,null
345,Khám phá Phú Quốc 1 ngày,"Núi,Văn hóa,Nghỉ dưỡng",3.1E7,2,5,5.0,1
404,Tham quan Hội An 4N3Đ,Mạo hiểm,2.8E7,1,4,null,null
411,Khám phá Ninh Bình 5N4Đ,Văn hóa,2.8E7,1,1,null,null
195,Khám phá Vũng Tàu cuối tuần,"Văn hóa,Sang trọng,Nghỉ dưỡng",2.4E7,2,4,3.0,1
299,Khám phá Hạ Long 1 ngày,"Văn hóa,Nghỉ dưỡng",2.1E7,1,2,3.0,1
452,Du lịch Quy Nhơn 2N1Đ,"Văn hóa,Mạo hiểm",2.1E7,1,5,null,null
221,Hành trình Hội An cuối tuần,"Núi,Nghỉ dưỡng",2.0E7,1,2,3.0,1
348,Hành trình Hội An 3N2Đ,"Văn hóa,Sang trọng",2.0E7,1,5,3.0,2
279,Nghỉ dưỡng Hội An 5N4Đ,Mạo hiểm,1.625E7,2,10,4.0,1


In [0]:
# Section 4: Payment Analysis - method split
# TODO: Analyze payment method, payment status, and total amount by method.
from pyspark.sql.functions import col, sum as _sum, round as _round

spark.sql("USE tourgo")

df_payment_method = spark.read.table("gold_payment_analysis")
df_payment_success = spark.read.table("gold_payment_success_rate")

total_payment_amount = df_payment_method.agg(_sum("total_amount").alias("total_amount")).collect()[0]["total_amount"] or 0
total_payment_count = df_payment_method.agg(_sum("count").alias("total_count")).collect()[0]["total_count"] or 0
avg_success_rate = df_payment_success.agg(_round(_sum(col("successful")) / _sum(col("total_attempts")) * 100, 2).alias("avg_success_rate")).collect()[0]["avg_success_rate"] or 0

print("=" * 60)
print("SECTION 4: PAYMENT ANALYSIS")
print("=" * 60)
print(f"Total Payment Amount : {total_payment_amount:>15,.0f} VND")
print(f"Payment Transactions : {total_payment_count:>15,.0f}")
print(f"Overall Success Rate : {avg_success_rate:>14.2f}%")

display(
    df_payment_method
    .select("payment_method", "status", "count", "total_amount")
    .orderBy("payment_method", "status")
)
# Databricks chart suggestion: pie chart, key = payment_method, value = count.

display(
    df_payment_success
    .select("payment_method", "total_attempts", "successful", "success_rate")
    .orderBy(col("success_rate").desc(), col("total_attempts").desc()))


SECTION 4: PAYMENT ANALYSIS
Total Payment Amount :     542,900,000 VND
Payment Transactions :              59
Overall Success Rate :         100.00%


payment_method,status,count,total_amount
BANK_TRANSFER,SUCCESS,16,1.38E8
CASH,SUCCESS,21,1.6255E8
VNPAY,SUCCESS,22,2.4235E8


Databricks visualization. Run in Databricks to view.

payment_method,total_attempts,successful,success_rate
VNPAY,22,22,100.0
CASH,21,21,100.0
BANK_TRANSFER,16,16,100.0


In [0]:
# Section 5: Customer Segments - K-Means result
# TODO: Show customer clusters, cluster size, and segment characteristics.
df_review_summary = spark.read.table("gold_review_summary")
df_rating_distribution = spark.read.table("gold_rating_distribution")

total_reviewed_tours = df_review_summary.count()
total_reviews = df_review_summary.agg(_sum("review_count").alias("total_reviews")).collect()[0]["total_reviews"] or 0
total_five_star = df_review_summary.agg(_sum("five_star").alias("five_star_reviews")).collect()[0]["five_star_reviews"] or 0

print("=" * 60)
print("SECTION 5: REVIEW AND RATING")
print("=" * 60)
print(f"Reviewed Tours       : {total_reviewed_tours:>15,}")
print(f"Total Reviews        : {total_reviews:>15,.0f}")
print(f"Five-star Reviews    : {total_five_star:>15,.0f}")

display(
    df_review_summary
    .select("tour_id", "title", "category_names", "avg_rating", "review_count", "five_star", "four_star", "one_star")
    .orderBy(col("avg_rating").desc(), col("review_count").desc())
    .limit(15)
)
# Databricks chart suggestion: bar chart, X = title, Y = avg_rating.

display(
    df_rating_distribution
    .select("rating", "count")
    .orderBy("rating"))


SECTION 5: REVIEW AND RATING
Reviewed Tours       :              45
Total Reviews        :              62
Five-star Reviews    :              22


tour_id,title,category_names,avg_rating,review_count,five_star,four_star,one_star
223,Du lịch Hội An 5N4Đ,"Núi,Sang trọng,Mạo hiểm",5.0,2,2,0,0
370,Tham quan Ninh Bình 1 ngày,"Núi,Văn hóa,Nghỉ dưỡng",5.0,2,2,0,0
332,Tham quan Quy Nhơn 1 ngày,"Biển,Núi,Sang trọng",5.0,2,2,0,0
314,Du lịch Hội An 5N4Đ,"Núi,Sang trọng",5.0,1,1,0,0
320,Tour cao cấp Đà Nẵng 4N3Đ,"Văn hóa,Mạo hiểm,Nghỉ dưỡng",5.0,1,1,0,0
345,Khám phá Phú Quốc 1 ngày,"Núi,Văn hóa,Nghỉ dưỡng",5.0,1,1,0,0
196,Tham quan Huế 5N4Đ,"Núi,Sang trọng",5.0,1,1,0,0
369,Du lịch Nha Trang 2N1Đ,"Biển,Núi,Mạo hiểm",5.0,1,1,0,0
199,Hành trình Cần Thơ 1 ngày,"Văn hóa,Sang trọng,Nghỉ dưỡng",5.0,1,1,0,0
351,Du lịch Phú Quốc 4N3Đ,"Biển,Núi,Sang trọng",5.0,1,1,0,0


Databricks visualization. Run in Databricks to view.

rating,count
3,21
4,19
5,22


In [0]:
# Section 6: Revenue Forecast - actual vs predicted
# TODO: Plot actual revenue vs predicted revenue and forecast error if available.

In [0]:
# Section 7: Anomaly Alerts - table
# TODO: Show revenue, booking, or payment anomalies with severity.

In [0]:
# Section 8: Real-Time Counter - streaming
# TODO: Show new booking counter, latest bookings, and time-window chart.
# Expected streaming path: s3://tourgo-data-lake/streaming/new_bookings/